## dataset

In [1]:
!pip install tqdm fastparquet

In [1]:
import requests
import yaml
import getpass
# import zstandard as zstd
import pandas as pd
import json
import io

from typing import Dict, List, Any
from tqdm import tqdm

In [2]:
df_queries = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_examples.parquet')
df_queries = df_queries[df_queries["product_locale"] == "us"]


In [3]:
df_queries.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train


In [4]:
unique_queries = df_queries["query"].drop_duplicates()

In [5]:
len(unique_queries)

97345

In [6]:
unique_queries.head(5)

0                                revent 80 cfm
16                !awnmower tires without rims
32                !qscreen fence without holes
149    # 10 self-seal envelopes without window
189                  # 2 pencils not sharpened
Name: query, dtype: object

In [7]:
random_queries = unique_queries.sample(n=1000, random_state=42)
len(random_queries)

1000

In [8]:
df_products = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet')
df_products = df_products[df_products["product_locale"] == "us"]
df_products.head(5)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
167168,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,None,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us
167169,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,None,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us
167170,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,None,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us
167171,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,None,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
167172,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,None,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us


In [9]:
df_random_queries = df_queries[
    df_queries["query"].isin(random_queries)
]

In [10]:
df = pd.merge(
    df_random_queries,
    df_products,
    how='left',
    left_on=['product_locale','product_id'],
    right_on=['product_locale', 'product_id']
)
df.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
0,3042,$5 items,100,B079HXXP4T,us,I,1,1,test,"Soft Scrub In-Tank Toilet Cleaner Duo-Cubes, A...",None,"Helps fight toilet ring, hard water, and limes...",Soft Scrub,Alpine Fresh
1,3043,$5 items,100,B07DZYGDS3,us,E,1,1,test,"Gillette Fusion5 Razors for Men, 1 Gillette Ra...",None,REFILLS FIT ALL GILLETTE 5-BLADE RAZOR HANDLES...,Gillette,None
2,3044,$5 items,100,B07HY9DC4N,us,E,1,1,test,"Summer's Eve Cleansing Cloths, Blissful Escape...",None,Summer's Eve Feminine Cleansing Wipes are safe...,Summer's Eve,None
3,3045,$5 items,100,B07M77RB97,us,E,1,1,test,BIC Flex 5 Hybrid Men's 5-Blade Disposable Raz...,None,"5 long lasting, flexible blades individually a...",BIC,Black
4,3046,$5 items,100,B07NTWYGJX,us,I,1,1,test,"6PCS Dual Heads Blackhead Remover, Pimple Come...",<b>About Our Factory:</b><br /> ✿✿Our factory ...,"♥ 【Dual Heads REMOVER】: 6PCS dual heads tools,...",USCOLOR,None


In [11]:
len(df)

18727

## index

In [13]:
SEARCH_INDEX = 'http://localhost:9200/myindex'

In [14]:
idx = requests.put(
    SEARCH_INDEX,
    json={
        "mappings": {
            "properties": {
                "name": {
                    "type": "text"
                },
                "description": {
                    "type": "text"
                }
            }
        }
    }
)
idx.json()

{'error': {'root_cause': [{'type': 'resource_already_exists_exception',
    'reason': 'index [myindex/X5nCAeD6TeS9kupkhnftAA] already exists',
    'index_uuid': 'X5nCAeD6TeS9kupkhnftAA',
    'index': 'myindex'}],
  'type': 'resource_already_exists_exception',
  'reason': 'index [myindex/X5nCAeD6TeS9kupkhnftAA] already exists',
  'index_uuid': 'X5nCAeD6TeS9kupkhnftAA',
  'index': 'myindex'},
 'status': 400}

In [15]:
def index_record(id, name, description):
    if id and name and description:
        try:
            return requests.post(
                f"{SEARCH_INDEX}/_doc/{id}",
                json={
                    'name': name,
                    'description': description    
                }
            )
        except:
            pass

In [16]:
for index, row in tqdm(df.iterrows(), total=len(df)):
    _ = index_record(row['example_id'], row['product_title'], row['product_description'])

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 18727/18727 [00:21<00:00, 861.57it/s]


In [17]:
response = requests.post(
    f"{SEARCH_INDEX}/_search",
    json={
        "size": 0,
        "track_total_hits": True
    }
)
response.json()

{'took': 283,
 'timed_out': False,
 'terminated_early': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 9295, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

In [18]:
def search_query(query='#$query##'):
    return {
      "query": {
        "multi_match": {
          "query": query,
          "fields": [f"name", "description"]
        }
      }
    }
    
def search(query):
    response = requests.post(
        f"{SEARCH_INDEX}/_search",
        json=search_query(query)
    )
    return response.json()

In [19]:
search('dinosaur')

{'took': 42,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 31, 'relation': 'eq'},
  'max_score': 13.017879,
  'hits': [{'_index': 'myindex',
    '_id': '1975317',
    '_score': 13.017879,
    '_source': {'name': 'Baby Dinosaur Balloon Set for Birthday Decor - 38 Inch, Pack of 4, 4D Dinosaur Foil Balloon | Kids Dinosaur Party Decorations | Dinosaur Balloons for Birthday Party | Dinosaur Birthday Party Supplies',
     'description': '<p><b>Are you a dinasaur fanatic?</b></p><p>Looking for a dinosaur theme party decorations for your kid</p> <p>We have got you covered with these beautiful and gigantic <b>Baby Dinosaur Foil Balloons </b> in two different style of dinosaur party balloons </p> <p>This dinosaur balloon kit is simply beautiful, gorgeous color to your dinosaur balloon birthday party as a backdrop for photos Booth.</p> <p>Configure it any way you wish; there are 1000s of ways to use your kit to build your b

## Quepid

### init

In [43]:
!docker compose run quepid-api-quepid bin/rake db:migrate
!docker compose run quepid-api-quepid bin/rake db:seed
!docker compose run quepid-api-quepid bundle exec thor user:create -a admin@example.com "Admin User" supersecret



WARN[0000] Found orphan containers (judge-student-quepid-api-quepid-run-7529f39cfef2, judge-student-quepid-api-quepid-run-456e18849e7e, judge-student-quepid-api-quepid-run-202ab1fbdcb7) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up. 
[+] create 2/2
 ✔ Container es-test                          Running                       0.0s
 ✔ Container judge-student-quepid-api-mysql-1 Running                       0.0s
[+] start 2/2
 ✔ Container es-test                          Running                       0.0s
 ✔ Container judge-student-quepid-api-mysql-1 Running                       0.0s
[+]  2/2
 ✔ Container es-test                          Running                       0.0s
 ✔ Container judge-student-quepid-api-mysql-1 Running                       0.0s
Container judge-student-quepid-api-mysql-1 Waiting 
Container es-test Waiting 
Container judge-student-quepid-api-mysql-1 Healthy 
Contain

In [44]:
!docker compose run quepid-api-quepid bundle exec thor user:add_api_key admin@example.com

WARN[0000] Found orphan containers (judge-student-quepid-api-quepid-run-34d775c45661, judge-student-quepid-api-quepid-run-952949a27991, judge-student-quepid-api-quepid-run-011750d7aac0, judge-student-quepid-api-quepid-run-7529f39cfef2, judge-student-quepid-api-quepid-run-456e18849e7e, judge-student-quepid-api-quepid-run-202ab1fbdcb7) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up. 
[+] create 2/2
 ✔ Container es-test                          Running                       0.0s
 ✔ Container judge-student-quepid-api-mysql-1 Running                       0.0s
[+] start 2/2
 ✔ Container es-test                          Running                       0.0s
 ✔ Container judge-student-quepid-api-mysql-1 Running                       0.0s
[+]  2/2
 ✔ Container es-test                          Running                       0.0s
 ✔ Container judge-student-quepid-api-mysql-1 Running                

### import esci

In [45]:
QUEPID_TOKEN="ee42fff36c32cf187ae30d59692e03e5e6dff04da4b09f9e6ec72767b58cfbeb"

In [46]:
AUTH = {
    "Authorization": f"Bearer {QUEPID_TOKEN}"
}

In [47]:
team = requests.post(
    'http://localhost:8081/api/teams/', 
    headers = AUTH,
    json={
        "name": "dredd"
    }   
)
team = team.json()

In [48]:
team

{'id': 1,
 'name': 'dredd',
 'created_at': '2026-09-06T09:44:31.429Z',
 'updated_at': '2026-09-06T09:44:31.429Z'}

In [49]:
endpoint = requests.post(
    'http://localhost:8081/api/search_endpoints/', 
    headers = AUTH,
    json={
        "name": "myindex",
        "endpoint_url": "http://quepid-api-elasticsearch:9200/myindex/_search",
        "search_engine": "es",
        "api_method": "POST",
        "proxy_requests": 1,   
    }   
)
endpoint = endpoint.json()

In [50]:
print(endpoint)

{'id': 1, 'name': 'myindex', 'owner': 1, 'search_engine': 'es', 'endpoint_url': 'http://quepid-api-elasticsearch:9200/myindex/_search', 'api_method': 'POST', 'custom_headers': None, 'archived': 0, 'created_at': '2026-09-06T09:44:50.060Z', 'updated_at': '2026-09-06T09:44:50.060Z', 'basic_auth_credential': None, 'mapper_code': None, 'proxy_requests': 1, 'options': None, 'requests_per_minute': None, 'test_query': None}


In [51]:
# list scorers
scorers = requests.get(
    'http://localhost:8081/api/scorers/', 
    headers = AUTH
)
print({s['id']: s['name'] for s in scorers.json()['items']})

{1: 'nDCG@10', 2: 'DCG@10', 3: 'CG@10', 4: 'P@10', 5: 'AP@10', 6: 'RR@10', 7: 'ERR@10'}


In [ ]:
book = requests.post(
    'http://localhost:8081/api/books/', 
    headers = AUTH,
    json={
        "name": "dredd",
        # No team_id: you belong to exactly one team, so the book joins it.
        # Pass "team_id": <id> if you are in several, or 0 to keep it unshared.

        # The ratings a judge may give. Quepid's UI copies these off the scorer
        # you pick when creating a book; nothing picks a scorer here, and a book
        # with no scale renders no rating buttons at all. ESCI's four labels in
        # descending relevance -- Exact, Substitute, Complement, Irrelevant --
        # so a judgement can be compared with esci_label directly.
        "scale": [0, 1, 2, 3],
        "scale_with_labels": {
            "0": "Irrelevant",
            "1": "Complement",
            "2": "Substitute",
            "3": "Exact",
        },
    }   
)
book.raise_for_status()
book = book.json()

In [ ]:
print(book)

In [52]:
case1 = requests.post(
    'http://localhost:8081/api/case/', 
    headers = AUTH,
    json={
        "name": "dredd",
        "scorer_id": 1,
        "book_id": book.get('id'),
        "search_endpoint_id": endpoint.get('id'),
        "search_query": json.dumps(search_query())
    }   
)
case1 = case1.json()

In [53]:
print(case1)

{'id': 1, 'case_name': 'dredd', 'last_try_number': 1, 'owner': 1, 'archived': 0, 'scorer_id': 1, 'created_at': '2026-09-06T09:45:02.250Z', 'updated_at': '2026-09-06T09:45:02.250Z', 'book_id': None, 'public': None, 'options': None, 'nightly': 1}


In [ ]:
# A book's queries are the distinct query_texts of its query/doc pairs, and a
# pair must carry a doc -- so the ground truth goes in whole, one pair per df
# row. doc_id is example_id, matching what index_record used as the
# Elasticsearch _id, so the book lines up with what a search on myindex returns
# (and joins straight back onto df, esci_label included).
df_pairs = df[df['product_title'].notna() & df['product_description'].notna()]

pairs = [
    {
        "query_text": row['query'],
        "doc_id": str(row['example_id']),
        "document_fields": {
            "name": row['product_title'],
            "description": row['product_description'],
        },
        # Nothing in query_doc_pairs holds a foreign id, so the ESCI query_id
        # travels in options. Deliberately not esci_label: the judge reads the
        # document, and the label is recoverable from doc_id anyway.
        "query_options": {"esci_query_id": int(row['query_id'])},
    }
    for _, row in df_pairs.iterrows()
]
print(f"{df_pairs['query'].nunique()} queries, {len(pairs)} pairs")

# Batched: a pair is identified by (query_text, doc_id), so re-running this
# cell adds nothing and a batch that fails can simply be sent again.
written = {"created": 0, "skipped": 0}
for i in tqdm(range(0, len(pairs), 500)):
    r = requests.post(
        f"http://localhost:8081/api/books/{book['id']}/query_doc_pairs/",
        headers = AUTH,
        json=pairs[i:i + 500]
    )
    r.raise_for_status()
    for k, v in r.json().items():
        written[k] += v

written

In [ ]:
# And now the labels. A pair carries no rating: the rating is a *judgement*, one
# row per (rater, pair), so ESCI's ground truth goes in as judgements of ours.
# The AI judge's verdicts then land on the same pairs under its own user id --
# SelectionStrategy offers a rater any pair it has not itself rated, up to three
# raters per pair -- which is what makes the two comparable afterwards.
ESCI_RATING = {"I": 0, "C": 1, "S": 2, "E": 3}   # the book's scale, set above

labels = [
    {
        "query_text": row['query'],
        "doc_id": str(row['example_id']),
        "rating": ESCI_RATING[row['esci_label']],
        "explanation": f"ESCI ground truth: {row['esci_label']}",
    }
    for _, row in df_pairs.iterrows()
]

# Identity is (rater, pair), so re-running updates rather than duplicating --
# unlike the pairs above, which are skipped. Rows naming a pair the book does
# not hold come back as "unknown" instead of failing the batch, so a non-zero
# count there means labels went nowhere and is worth looking at.
written = {"created": 0, "updated": 0, "unchanged": 0, "unknown": 0}
for i in tqdm(range(0, len(labels), 500)):
    r = requests.post(
        f"http://localhost:8081/api/books/{book['id']}/judgements/",
        headers = AUTH,
        json=labels[i:i + 500]
    )
    r.raise_for_status()
    for k, v in r.json().items():
        written[k] += v

written

### AI Judge

In [ ]:
import os

judge = requests.post(
    'http://localhost:8081/api/ai_judges/',
    headers = AUTH,
    json={
        "name": "dredd",
        "llm_key": os.environ.get("OPENAI_API_KEY") or getpass.getpass("LLM API key: "),
        "judge_options": {"llm_model": "gpt-4o"},
    }
)
judge.raise_for_status()
judge = judge.json()
judge['id'], judge['name'], judge['judge_options']

In [70]:
# Sharing a team is not enough to make the judge usable on the book.
attached = requests.post(
    f"http://localhost:8081/api/ai_judges/{judge['id']}/books/",
    headers = AUTH,
    json={"book_id": book['id']}
)
attached.raise_for_status()

# # Idempotent, so re-running this cell is a no-op -- unlike the cell above,
# # which has no natural key and would make a second judge.
# [j['name'] for j in requests.get(
#     'http://localhost:8081/api/ai_judges/', headers = AUTH
# ).json()['items']]

NameError: name 'judge' is not defined

In [71]:
# Nothing here starts the judging run: run_judge_judy is an HTML route that
# enqueues RunJudgeJudyJob, and neither Quepid's API nor this one exposes it.
# Open the book, pick the judge, and let it work through the 9295 pairs:
print(f"http://localhost:3000/books/{book['id']}/judgement_stats")

NameError: name 'book' is not defined

## train student

In [ ]:
compare

In [ ]:
human labels - better teacher - dspy??